# The Keyboard Tax: GPU run on Google Colab

Runs step 4 (translate) and step 5 (score) on a free Colab GPU.

**Before you start:** menu **Runtime → Change runtime type → T4 GPU → Save**.

Colab's disk is temporary: when the session ends, its files are gone. So each part ends
with a **download** of the results to your computer, and part B starts by uploading them.

- **Part A (sections 4 to 9):** translate. Needs your Hugging Face token.
- **Part B (sections 10 to 13):** score. Needs the zip file from part A.

Always run sections 1 to 3 first.

## 1. Check the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))

## 2. Get the code

After the pull request is merged, change `BRANCH` to `main`.

In [ ]:
BRANCH = "coder2/steps-4-6"

%cd /content
!rm -rf Keyboard_Tax
!git clone -q --branch {BRANCH} https://github.com/karoli-rey/Keyboard_Tax.git
%cd Keyboard_Tax
!git log --oneline -1

## 3. Install the packages

Colab already has PyTorch and transformers. We add only what is missing.

In [ ]:
!pip install -q "spylls==0.1.7" "wordfreq==3.1.1" datasets sacremoses accelerate bitsandbytes
!pip install -q -e . --no-deps

# Part A: translate (step 4)

## 4. Log in to Hugging Face

Accept the FLORES+ terms once at https://huggingface.co/datasets/openlanguagedata/flores_plus
and paste your own token (role "Read") in the box. It is not saved in this notebook.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 5. Steps 1 to 3

Expected at the end: 1012 sentences; `clean` and `stripped_100` 1012 each;
`a_only`, `b_only` and `noise` 422 each.

In [ ]:
!bash scripts/get_dictionary.sh
!python -m keyboard_tax.data
!python -m keyboard_tax.classify
!python -m keyboard_tax.versions

## 6. Try 20 sentences first

Read a few translations before the full run.

In [ ]:
!python -m keyboard_tax.translate --limit 20 --systems opus_mt nllb

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 120)
tried = pd.read_csv("outputs/4_translations.csv")
tried[tried["sent_id"] == tried["sent_id"].min()]

## 7. Full run

First the two translation models (a few minutes), then the LLM (about 20 minutes plus the
download of the model). Download the results after each cell (section 9).

In [ ]:
!python -m keyboard_tax.translate --systems opus_mt nllb

In [ ]:
!python -m keyboard_tax.translate --systems llm

## 8. Check the result

Expected: 3290 rows per system, 0 empty translations, 0 LLM answers that start with a chat
phrase or have more than one line.

In [ ]:
import pandas as pd

table = pd.read_csv("outputs/4_translations.csv")
print(table.groupby("system").size())
print("empty translations:", int(table["hyp"].fillna("").str.strip().eq("").sum()))
llm = table[table["system"] == "llm"]
answers = llm["hyp"].fillna("")
chatty = answers.str.contains(
    r"^\W*(?:here is|here's|sure|certainly|translation|english)\b", case=False, regex=True
)
print("LLM answers starting with a chat phrase:", int(chatty.sum()))
print("LLM answers with more than one line:", int(answers.str.contains("\n").sum()))
other_script = answers.str.contains(r"[\u3040-\u30ff\u3400-\u9fff\uac00-\ud7af]", regex=True)
print("LLM answers with Chinese, Japanese or Korean characters:", int(other_script.sum()))

## 9. Download the results

Saves `outputs_backup.zip` on your computer. Keep it: part B and step 6 need it.
Unzip it into `Keyboard_Tax/` on your laptop so the files land in `Keyboard_Tax/outputs/`.

In [ ]:
from google.colab import files

!cd /content/Keyboard_Tax && zip -qr /content/outputs_backup.zip outputs
files.download("/content/outputs_backup.zip")

# Part B: score (step 5)

## 10. Upload the translations

Only needed in a new session (after sections 1 to 3): choose the `outputs_backup.zip` from
part A. Skip this cell if you just ran part A in this same session.

In [ ]:
import zipfile

from google.colab import files

for name in files.upload():
    with zipfile.ZipFile(name) as archive:
        archive.extractall("/content/Keyboard_Tax")
!ls /content/Keyboard_Tax/outputs

## 11. Install the scoring packages

In [ ]:
!pip install -q sacrebleu unbabel-comet

## 12. Score

chrF and COMET for all 9870 translations, then BLEU, chrF and mean COMET per version and
system. COMET downloads its model first (about 2 GB). Copy the two sacreBLEU signature lines
at the end into `notes/decisions.md`.

In [ ]:
!python -m keyboard_tax.score

## 13. Download the results

The zip now also holds `5_scores.csv` and `5_corpus_scores.csv`.

In [ ]:
from google.colab import files

!cd /content/Keyboard_Tax && zip -qr /content/outputs_backup.zip outputs
files.download("/content/outputs_backup.zip")